# ⚡ ProcessSigma — Delivery Intelligence Pipeline
## Minimal Example | Six Sigma + Data Engineering + Process Mining

> **What this notebook demonstrates:**
> A complete — but lightweight — taste of the ProcessSigma framework.
> From raw CSV data to Six Sigma indicators, Statistical Process Control,
> and a live Streamlit dashboard. No database required.

---

### How to use this notebook

1. Place `template_injecao.csv` in the `data/` folder (or use the synthetic generator below)
2. Run all cells sequentially (`Cell → Run All`)
3. At the end, launch the Streamlit app with `streamlit run minimal_example_app.py`

### Stack used in this example
- **Python** — ETL and Six Sigma calculations
- **pandas** — data manipulation
- **plotly** — interactive charts
- **Streamlit** — dashboard (launched at the end)


## 0. Setup — Install Dependencies

In [15]:
# Run this cell only once
# !pip install pandas plotly streamlit openpyxl loguru
print("Dependencies ready. If not installed, uncomment the line above.")


Dependencies ready. If not installed, uncomment the line above.


## 1. Imports and Configuration

In [16]:
import pandas as pd
import numpy as np
import math
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from loguru import logger
import os
import subprocess
import sys
from export_graphs import save_figure, save_figures

# Output folder for exports
Path("exports").mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)

print("✅ Imports OK")

✅ Imports OK


## 2. Data Ingestion — Bronze Layer

The Bronze layer stores data **exactly as it arrives** from the source — no transformations, no filtering.
This guarantees full traceability and the ability to reprocess from scratch at any time.

> **Try it:** edit `template_injecao.csv` and re-run the notebook to see indicators update automatically.


In [17]:
DATA_PATH = "data/template_injecao.csv"

# ── If CSV not found, generate synthetic data ────────────────
if not Path(DATA_PATH).exists():
    logger.warning(f"{DATA_PATH} not found. Generating synthetic data...")
    np.random.seed(42)
    n = 500
    regions = ["NORTH AMERICA", "SOUTH AMERICA", "EUROPE", "ASIA", "OCEANIA"]
    categories = ["Electronics", "Clothing", "Sports", "Home", "Food"]

    synthetic = pd.DataFrame({
        "Order Id": range(10001, 10001 + n),
        "order date (DateOrders)": pd.date_range("2022-01-01", periods=n, freq="12h").strftime("%m/%d/%Y %H:%M"),
        "shipping date (DateOrders)": (pd.date_range("2022-01-05", periods=n, freq="12h") +
                                        pd.to_timedelta(np.random.randint(-1, 5, n), unit="D")).strftime("%m/%d/%Y %H:%M"),
        "Days for shipment (scheduled)": np.random.randint(3, 7, n),
        "Days for shipping (real)": np.random.randint(2, 9, n),
        "Delivery Status": np.random.choice(
            ["Late delivery", "Advance shipping", "Shipping on time"], n, p=[0.57, 0.20, 0.23]),
        "Late_delivery_risk": np.random.randint(0, 2, n),
        "Order Region": np.random.choice(regions, n),
        "Order Country": np.random.choice(["USA", "Brazil", "Germany", "Japan", "Australia"], n),
        "Category Name": np.random.choice(categories, n),
    })
    synthetic.to_csv(DATA_PATH, index=False)
    logger.success(f"Synthetic data generated: {n} records → {DATA_PATH}")

# ── Load Bronze ──────────────────────────────────────────────
COLUMNS_MAP = {
    "Order Id": "order_id",
    "order date (DateOrders)": "order_date",
    "shipping date (DateOrders)": "ship_date",
    "Days for shipment (scheduled)": "days_scheduled",
    "Days for shipping (real)": "days_real",
    "Delivery Status": "delivery_status",
    "Late_delivery_risk": "late_delivery_risk",
    "Order Region": "order_region",
    "Order Country": "order_country",
    "Category Name": "category_name",
}

bronze = pd.read_csv(DATA_PATH, encoding="latin1")
bronze = bronze[[c for c in COLUMNS_MAP if c in bronze.columns]].rename(columns=COLUMNS_MAP)
bronze["damage_flag"] = False
bronze["return_flag"] = False
bronze["source"] = "csv_upload"
bronze["ingested_at"] = pd.Timestamp.now()

print(f"🥉 Bronze layer: {len(bronze):,} records loaded")
bronze.head(3)


🥉 Bronze layer: 500 records loaded


,order_id,order_date,ship_date,days_scheduled,days_real,delivery_status,late_delivery_risk,order_region,order_country,category_name,damage_flag,return_flag,source,ingested_at
0,10001,01/01/2022 00:00,01/07/2022 00:00,3,5,Late delivery,1,SOUTH AMERICA,Brazil,Sports,False,False,csv_upload,2026-04-06 17:05:25.259240
1,10002,01/01/2022 12:00,01/08/2022 12:00,4,2,Shipping on time,0,ASIA,Australia,Sports,False,False,csv_upload,2026-04-06 17:05:25.259240
2,10003,01/02/2022 00:00,01/07/2022 00:00,4,4,Shipping on time,0,OCEANIA,Brazil,Clothing,False,False,csv_upload,2026-04-06 17:05:25.259240


## 3. Transformation — Silver Layer

The Silver layer **cleans, standardizes, and enriches** the data.
Two calculated fields are created that power all Six Sigma analysis:

- **`delay_days`** = days_real − days_scheduled (negative = early, positive = late)
- **`is_late`** = True if delay_days > 0


In [18]:
# Parse dates
bronze["order_date"] = pd.to_datetime(bronze["order_date"], errors="coerce")
bronze["ship_date"]  = pd.to_datetime(bronze["ship_date"],  errors="coerce")

# Calculate key fields
silver = bronze.copy()
silver["delay_days"] = silver["days_real"] - silver["days_scheduled"]
silver["is_late"]    = silver["delay_days"] > 0
silver["period"]     = silver["order_date"].dt.to_period("M").dt.to_timestamp()

# Clean
silver = silver.dropna(subset=["order_id", "order_date", "delay_days"])
silver = silver.drop_duplicates(subset=["order_id"])

print(f"🥈 Silver layer: {len(silver):,} unique orders")
print(f"   Late deliveries: {silver['is_late'].sum():,} ({silver['is_late'].mean()*100:.1f}%)")
print(f"   Mean delay:      {silver['delay_days'].mean():.2f} days")
print(f"   Std deviation:   {silver['delay_days'].std():.2f} days")
silver[["order_id","order_date","delay_days","is_late","order_region"]].head(3)


🥈 Silver layer: 500 unique orders
   Late deliveries: 260 (52.0%)
   Mean delay:      0.64 days
   Std deviation:   2.43 days


,order_id,order_date,delay_days,is_late,order_region
0,10001,2022-01-01 00:00:00,2,True,SOUTH AMERICA
1,10002,2022-01-01 12:00:00,-2,False,ASIA
2,10003,2022-01-02 00:00:00,0,False,OCEANIA


## 4. Aggregation — Gold Layer

The Gold layer produces **ready-to-consume indicators**.
This is what the dashboard and Power BI consume directly.


In [19]:
def dpmo_to_sigma(dpmo: float) -> float:
    if dpmo <= 0:       return 6.0
    if dpmo >= 999999:  return 0.5
    if dpmo >= 500000:  return 1.5
    if dpmo >= 308538:  return 2.0
    if dpmo >= 66807:   return 3.0
    if dpmo >= 6210:    return 4.0
    if dpmo >= 233:     return 5.0
    x = 29.37 - 2.221 * math.log(dpmo)
    return round(0.8406 + math.sqrt(max(0, x)), 2) if x > 0 else 0.5

def classify_sigma(sigma: float) -> str:
    if sigma >= 6: return "🏆 World Class"
    if sigma >= 5: return "✅ Excellent"
    if sigma >= 4: return "🟡 Good"
    if sigma >= 3: return "🟠 Attention"
    return "🔴 Critical"

# ── Gold OTD ─────────────────────────────────────────────────
gold_otd = []
for (period, region), grp in silver.groupby(["period", "order_region"]):
    total   = len(grp)
    on_time = int((~grp["is_late"]).sum())
    gold_otd.append({
        "period": period, "region": region,
        "total_deliveries": total, "on_time": on_time,
        "otd_pct": round(on_time / total * 100, 2)
    })
gold_otd = pd.DataFrame(gold_otd)

# ── Gold Sigma ────────────────────────────────────────────────
gold_sigma = []
for period, grp in silver.groupby("period"):
    opps    = len(grp)
    defects = int(grp["is_late"].sum() + grp["damage_flag"].sum() + grp["return_flag"].sum())
    dpmo    = round(defects / opps * 1_000_000, 2)
    sigma   = dpmo_to_sigma(dpmo)
    gold_sigma.append({
        "period": period, "total_opportunities": opps,
        "total_defects": defects, "dpmo": dpmo,
        "sigma_level": sigma, "classification": classify_sigma(sigma)
    })
gold_sigma = pd.DataFrame(gold_sigma)

# ── Summary ───────────────────────────────────────────────────
global_otd   = round(gold_otd["on_time"].sum() / gold_otd["total_deliveries"].sum() * 100, 2)
global_dpmo  = round(silver["is_late"].sum() / len(silver) * 1_000_000, 2)
global_sigma = dpmo_to_sigma(global_dpmo)

print(f"🥇 Gold layer ready")
print(f"   Global OTD:      {global_otd}%")
print(f"   Global DPMO:     {global_dpmo:,.0f}")
print(f"   Sigma Level:     {global_sigma}σ  {classify_sigma(global_sigma)}")
print(f"   Target (4σ):     6,210 DPMO | 95% OTD")


🥇 Gold layer ready
   Global OTD:      48.0%
   Global DPMO:     520,000
   Sigma Level:     1.5σ  🔴 Critical
   Target (4σ):     6,210 DPMO | 95% OTD


## 5. Exploratory Analysis

Three charts that tell the Business Case story:
1. **Delay distribution** — is the process systematically late?
2. **OTD by region** — is there a geographic pattern?
3. **Sigma evolution** — is the process improving over time?


In [20]:
# 1. Garante que o diretório "exports" exista na raiz
output_dir = "exports"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Chart 1: Delay distribution
fig1 = px.histogram(
    silver, x="delay_days", nbins=20,
    title="Lead Time Deviation Distribution",
    labels={"delay_days": "Deviation (days)", "count": "Frequency"},
    color_discrete_sequence=["#4fc3f7"]
)
fig1.add_vline(
    x=0,
    line_dash="dash",
    line_color="red",
    annotation_text="Zero deviation (on-time)",
    annotation_position="top left"
)
fig1.add_vline(
    x=silver["delay_days"].mean(),
    line_dash="dot",
    line_color="orange",
    annotation_text=f"Mean: {silver['delay_days'].mean():.2f}d",
    annotation_position="top right"
)

# 2. Exportação automática (pode ser .png, .jpg, .pdf ou .svg)
saved_path = save_figure(fig1, "delay_distribution.png")
print(f"✅ Gráfico salvo em: {saved_path}")

fig1.show()

✅ Gráfico salvo em: exports\delay_distribution.png


In [21]:
# Chart 2: OTD by region
otd_region = silver.groupby("order_region").agg(
    total=("order_id","count"), late=("is_late","sum")
).reset_index()
otd_region["otd_pct"] = round((otd_region["total"] - otd_region["late"]) / otd_region["total"] * 100, 2)
otd_region = otd_region.sort_values("otd_pct")

fig2 = px.bar(
    otd_region, x="otd_pct", y="order_region", orientation="h",
    title="OTD (%) by Region",
    labels={"otd_pct": "OTD (%)", "order_region": "Region"},
    color="otd_pct",
    color_continuous_scale=["red","yellow","green"],
    range_color=[40, 100]
)
fig2.add_vline(x=95, line_dash="dash", line_color="green",
               annotation_text="Target 95%")
fig2.add_vline(x=otd_region["otd_pct"].mean(), line_dash="dot",
               line_color="white",
               annotation_text=f"Mean {otd_region['otd_pct'].mean():.1f}%")

# Salvar gráfico
saved_path = save_figure(fig2, "otd_by_region.png")
print(f"✅ Gráfico salvo em: {saved_path}")

fig2.show()

✅ Gráfico salvo em: exports\otd_by_region.png


In [22]:
# Chart 3: Sigma evolution
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=gold_sigma["period"], y=gold_sigma["sigma_level"],
    mode="lines+markers", name="Sigma Level",
    line=dict(color="#4fc3f7", width=2),
    marker=dict(size=7, color=gold_sigma["sigma_level"],
                colorscale=["red","orange","yellow","green"],
                cmin=1, cmax=5)
))
fig3.add_hrect(y0=0,   y1=3, fillcolor="red",    opacity=0.07, line_width=0)
fig3.add_hrect(y0=3,   y1=4, fillcolor="orange", opacity=0.07, line_width=0)
fig3.add_hrect(y0=4,   y1=5, fillcolor="yellow", opacity=0.07, line_width=0)
fig3.add_hrect(y0=5,   y1=6, fillcolor="green",  opacity=0.07, line_width=0)
fig3.add_hline(y=4, line_dash="dash", line_color="green",
               annotation_text="Target 4σ (Good)")
fig3.update_layout(
    title="Monthly Sigma Level Evolution",
    xaxis_title="Period", yaxis_title="Sigma Level (σ)",
    yaxis=dict(range=[0, 6.5])
)

# Salvar gráfico
saved_path = save_figure(fig3, "sigma_evolution.png")
print(f"✅ Gráfico salvo em: {saved_path}")

fig3.show()

✅ Gráfico salvo em: exports\sigma_evolution.png


## 6. Statistical Process Control (SPC) — P-Chart

The P-Chart monitors the **proportion of defects** over time.
It answers: is the process statistically stable, or are there special causes?

> **Key insight:** a process can be stable AND out of specifications simultaneously.
> That is the most dangerous scenario — the problem is chronic, not accidental.


In [23]:
# P-Chart manual (without qcc — pure Python)
monthly = silver.groupby("period").agg(
    total=("order_id","count"), late=("is_late","sum")
).reset_index()
monthly["prop"] = monthly["late"] / monthly["total"]

p_bar = monthly["late"].sum() / monthly["total"].sum()
monthly["ucl"] = p_bar + 3 * np.sqrt(p_bar * (1 - p_bar) / monthly["total"])
monthly["lcl"] = (p_bar - 3 * np.sqrt(p_bar * (1 - p_bar) / monthly["total"])).clip(lower=0)
monthly["out_of_control"] = (monthly["prop"] > monthly["ucl"]) | (monthly["prop"] < monthly["lcl"])

fig_p = go.Figure()
fig_p.add_trace(go.Scatter(
    x=monthly["period"], y=monthly["ucl"],
    mode="lines", name="UCL", line=dict(color="red", dash="dash")))
fig_p.add_trace(go.Scatter(
    x=monthly["period"], y=monthly["lcl"],
    mode="lines", name="LCL", line=dict(color="red", dash="dash"),
    fill="tonexty", fillcolor="rgba(255,0,0,0.05)"))
fig_p.add_trace(go.Scatter(
    x=monthly["period"], y=monthly["prop"],
    mode="lines+markers", name="Defect Rate",
    line=dict(color="#4fc3f7"),
    marker=dict(
        size=8,
        color=["red" if v else "#4fc3f7" for v in monthly["out_of_control"]]
    )
))
fig_p.add_hline(y=p_bar, line_color="white", line_dash="solid",
                annotation_text=f"CL = {p_bar:.3f} ({p_bar*100:.1f}%)")

out_count = monthly["out_of_control"].sum()
fig_p.update_layout(
    title=f"P-Chart — Monthly Defect Rate | {out_count} point(s) out of control",
    xaxis_title="Period", yaxis_title="Proportion of Defects"
)

# Salvar gráfico
saved_path = save_figure(fig_p, "p_chart_defect_rate.png")
print(f"✅ Gráfico salvo em: {saved_path}")

fig_p.show()

print(f"Center Line (CL): {p_bar:.4f} ({p_bar*100:.2f}%)")
print(f"Points out of control: {out_count}")
if out_count == 0:
    print("→ Process is STABLE (but check if CL meets your target!)")
else:
    print("→ Special causes detected — investigate flagged periods.")

✅ Gráfico salvo em: exports\p_chart_defect_rate.png


Center Line (CL): 0.5200 (52.00%)
Points out of control: 0
→ Process is STABLE (but check if CL meets your target!)


## 7. Executive Summary — Business Case

This section consolidates all findings into a decision-ready summary.


In [24]:
print("=" * 60)
print("PROCESSSIGMA — EXECUTIVE SUMMARY")
print("=" * 60)
print(f"\nDataset: {len(silver):,} unique orders analyzed")
print(f"Periods: {silver['period'].nunique()} months")
print(f"Regions: {silver['order_region'].nunique()}")

print("\n── CURRENT STATE ──────────────────────────────────────")
print(f"  Global OTD:    {global_otd}%   (Target: >= 95%)")
print(f"  DPMO:          {global_dpmo:,.0f}  (Target: <= 6,210)")
print(f"  Sigma Level:   {global_sigma}σ         (Target: >= 4.0σ)")
print(f"  Classification: {classify_sigma(global_sigma)}")

gap_pct = round((global_dpmo - 6210) / global_dpmo * 100, 1) if global_dpmo > 6210 else 0
print(f"\n  Gap to target: {gap_pct}% DPMO reduction required")

print("\n── RECOMMENDATION ─────────────────────────────────────")
if global_sigma < 2:
    print("  🔴 CRITICAL: Process redesign required.")
    print("     Root cause investigation mandatory before any fix.")
elif global_sigma < 3:
    print("  🟠 ATTENTION: Significant improvement needed.")
    print("     Apply DMAIC methodology to identify root causes.")
elif global_sigma < 4:
    print("  🟡 BELOW TARGET: Process improvement project recommended.")
    print("     Focus on reducing variability in worst-performing regions.")
else:
    print("  🟢 ON TARGET: Process is capable. Focus on sustaining gains.")
print("=" * 60)
print("\nNext step: run the Streamlit dashboard")
print("  streamlit run minimal_example_app.py")


PROCESSSIGMA — EXECUTIVE SUMMARY

Dataset: 500 unique orders analyzed
Periods: 9 months
Regions: 5

── CURRENT STATE ──────────────────────────────────────
  Global OTD:    48.0%   (Target: >= 95%)
  DPMO:          520,000  (Target: <= 6,210)
  Sigma Level:   1.5σ         (Target: >= 4.0σ)
  Classification: 🔴 Critical

  Gap to target: 98.8% DPMO reduction required

── RECOMMENDATION ─────────────────────────────────────
  🔴 CRITICAL: Process redesign required.
     Root cause investigation mandatory before any fix.

Next step: run the Streamlit dashboard
  streamlit run minimal_example_app.py


## 8. Launch the Dashboard

Run the cell below to start the Streamlit app,
or execute in your terminal:

```bash
streamlit run minimal_example_app.py
```

The dashboard connects to the same data processed in this notebook
and provides interactive filters, real-time charts, and downloadable reports.


In [25]:
import subprocess, sys

print("Launching Streamlit dashboard...")
print("If it does not open automatically, visit: http://localhost:8501")
print("Press Ctrl+C in the terminal to stop.")

# Uncomment to launch directly from the notebook:
# subprocess.Popen([sys.executable, "-m", "streamlit", "run", "minimal_example_app.py"])
print("\nTo launch, run in your terminal:")
print("  streamlit run minimal_example_app.py")


Launching Streamlit dashboard...
If it does not open automatically, visit: http://localhost:8501
Press Ctrl+C in the terminal to stop.

To launch, run in your terminal:
  streamlit run minimal_example_app.py


In [27]:
# ── Salvar todos os gráficos de uma vez ──────────────────────
print("\n📊 Salvando todos os gráficos...")
all_figures = {
    "delay_distribution.png": fig1,
    "otd_by_region.png": fig2,
    "sigma_evolution.png": fig3,
    "p_chart_defect_rate.png": fig_p,
}

saved_files = save_figures(all_figures)
print("✅ Todos os gráficos salvos:")
for filename, path in saved_files.items():
    print(f"   • {filename} → {path}")

print(f"\n📁 Verifique a pasta 'exports/' para ver todos os arquivos salvos.")


📊 Salvando todos os gráficos...
✅ Todos os gráficos salvos:
   • delay_distribution.png → exports\delay_distribution.png
   • otd_by_region.png → exports\otd_by_region.png
   • sigma_evolution.png → exports\sigma_evolution.png
   • p_chart_defect_rate.png → exports\p_chart_defect_rate.png

📁 Verifique a pasta 'exports/' para ver todos os arquivos salvos.
